In [142]:
import numpy as np
from scipy.sparse import eye, lil_matrix, diags
from resource_estimate_utils import *
from os.path import join
from time import time
import networkx as nx

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.synthesis import LieTrotter, SuzukiTrotter
from qiskit import transpile
from qiskit.circuit.library import PauliEvolutionGate
from pytket import OpType
from pytket.passes import RemoveRedundancies, CommuteThroughMultis, SequencePass, FullPeepholeOptimise, auto_rebase_pass
from pytket.extensions.qiskit import qiskit_to_tk


from os.path import join, dirname
from utils import *
import matplotlib.pyplot as plt

In [194]:
# def lchs_sim_circuit(n, k1, k2, t, J, h, gamma, r=1):
#     assert k1 != k2

#     circuit = QuantumCircuit(n+1)

#     dt = t / r
#     # Second-order Trotter
#     for _ in range(r):
#         # TFIM part
#         for i in range(n):
#             circuit.rx(dt * h, 1 + i)
#         for i in range(n-1):
#             circuit.rzz(dt * J, 1 + i, 1 + i + 1)

#         # Anti-Hermitian part
#         a = -2 * dt * gamma * (k1 + k2) / 2
#         b = -2 * dt * gamma * (k1 - k2) / 2
#         for i in range(n):

#             circuit.rz(-b, 0)
#             circuit.rz(a, 1 + i)
#             circuit.h(0)
#             circuit.h(1 + i)
#             circuit.rxx(b, 0, 1 + i)
#             circuit.h(0)
#             circuit.h(1 + i)
#         # TFIM part
#         for i in range(n):
#             circuit.rx(dt * h, 1 + i)
#         for i in range(n-1):
#             circuit.rzz(dt * J, 1 + i, 1 + i + 1)
            
#     return circuit

def get_lchs_hamiltonian(n, J, h, gamma, k1, k2):

    pauli_op_list = []
    # Hermitian part
    for i in range(n):
        op = (n+1) * ['I']
        op[i] = 'X'
        pauli_op_list.append((''.join(op), h))
    for i in range(n-1):
        op = (n+1) * ['I']
        op[i] = 'Z'
        op[i+1] = 'Z'
        pauli_op_list.append((''.join(op), J))

    # Anti-Hermitian part
    for i in range(n):
        '''Controlled on \ket{0}'''
        op = (n+1) * ['I']
        op[i] = 'Z'
        op[n] = 'I'
        pauli_op_list.append((''.join(op), -k1 * gamma / 2))

        op = (n+1) * ['I']
        op[i] = 'I'
        op[n] = 'Z'
        pauli_op_list.append((''.join(op), k1 * gamma / 2))

        op = (n+1) * ['I']
        op[i] = 'Z'
        op[n] = 'Z'
        pauli_op_list.append((''.join(op), -k1 * gamma / 2))

        '''Controlled on \ket{1}'''
        op = (n+1) * ['I']
        op[i] = 'Z'
        op[n] = 'I'
        pauli_op_list.append((''.join(op), -k2 * gamma / 2))

        op = (n+1) * ['I']
        op[i] = 'I'
        op[n] = 'Z'
        pauli_op_list.append((''.join(op), -k2 * gamma / 2))

        op = (n+1) * ['I']
        op[i] = 'Z'
        op[n] = 'Z'
        pauli_op_list.append((''.join(op), k2 * gamma / 2))

    return pauli_op_list

def get_xi_pauli_op(n_p, R):
    xi_pauli_list = []
    op = n_p * ['I']
    xi_pauli_list.append((''.join(op), 1))

    for i in range(n_p - 1):
        op = n_p * ['I']
        op[n_p-1-i] = 'Z'
        xi_pauli_list.append((''.join(op), 2 ** i))
    
    op = n_p * ['I']
    op[0] = 'Z'
    xi_pauli_list.append((''.join(op), - 2 ** (n_p - 1)))

    xi_pauli_op = 0.5 * (2 * np.pi / (2 * R)) * SparsePauliOp.from_list(xi_pauli_list)
    return xi_pauli_op

def get_schrodingerization_hamiltonian(n, J, h, gamma, n_p, R):

    H_1_pauli_list = []
    H_2_pauli_list = []

    # Hermitian part
    for i in range(n):
        op = n * ['I']
        op[i] = 'X'
        H_2_pauli_list.append((''.join(op), -h))
    for i in range(n-1):
        op = n * ['I']
        op[i] = 'Z'
        op[i+1] = 'Z'
        H_2_pauli_list.append((''.join(op), -J))
    
    # Anti-Hermitian part
    for i in range(n):
        op = n * ['I']
        H_1_pauli_list.append((''.join(op), -gamma))
        op = n * ['I']
        op[i] = 'Z'
        H_1_pauli_list.append((''.join(op), gamma))

    xi_pauli_list = get_xi_pauli_op(n_p, R).to_list()

    pauli_op_list = []
    for i in range(len(H_1_pauli_list)):
        for j in range(len(xi_pauli_list)):
            pauli_op_list.append((H_1_pauli_list[i][0] + xi_pauli_list[j][0], -H_1_pauli_list[i][1] * xi_pauli_list[j][1]))
    for i in range(len(H_2_pauli_list)):
        pauli_op_list.append((H_2_pauli_list[i][0] + ''.join(n_p * ['I']), -H_2_pauli_list[i][1]))

    return pauli_op_list

LCHS

In [ ]:
def estimate_trotter_error(n, T, pauli_op, num_samples):
    max_error = 0
    H = pauli_op.to_matrix(sparse=True)
    for _ in range(num_samples):
        psi_0 = np.random.randn(2 ** n) + 1j * np.random.randn(2 ** n)
        psi_true = expm_multiply(-1j * H * T, psi_0)

        # Compute Trotter error
        # psi_trot = expm_multiply(-1j * H_trot * T, psi_0)
        error = norm(psi_true - psi_trot, ord=2)
        if error > max_error:
            max_error = error
        
    return max_error

In [195]:

T = 1
J = 1
h = 1
gamma = 0.05
R = 128
n_p = 8
N_p = 2 ** n_p
print(f"N_p={N_p}")

error_tol = 5e-2
trotter_method = "second_order"


n = 2

k1 = R
k2 = -R

pauli_op_list = get_lchs_hamiltonian(n, J, h, gamma, k1, k2)
pauli_op = SparsePauliOp.from_list(pauli_op_list)
H = pauli_op.to_matrix(sparse=True)


PauliEvolutionGate(pauli_op)

N_p=256


Instruction(name='PauliEvolution', num_qubits=3, num_clbits=0, params=[1.0])

Schrodingerization

In [ ]:

T = 1
J = 1
h = 1
gamma = 0.05
R = 128
n_p = 4
N_p = 2 ** n_p
print(f"N_p={N_p}")

error_tol = 5e-2
trotter_method = "second_order"


n = 2

k1 = R
k2 = -R

r_vals = np.arange(1, 100, 2)
trot_error = []
trot_error_2 = []
for r in r_vals:
    pauli_op = SparsePauliOp.from_list(get_schrodingerization_hamiltonian(n, J, h, gamma, n_p, R))
    circuit = SuzukiTrotter(order=2, reps=r).synthesize(PauliEvolutionGate((T * pauli_op).group_commuting()))

    J_mat = np.zeros((n,n))
    for i in range(n-1):
        J_mat[i,i+1] = J
    H2 = -(sum_J_zz(n, J_mat) + sum_h_x(n, h * np.ones(n)))

    H1 = - sum_delta_n(n, 2 * gamma * np.ones(n))

    H_mine = - (kron(H1, -diags(np.fft.fftfreq(N_p, d = 1/(N_p * 2 * np.pi / (2 * R))))) + kron(H2, eye(N_p)))


    trot_error.append(np.linalg.norm(expm(-1j * T * H_mine) - np.array(Operator(circuit)), ord=2))
plt.plot(r_vals, trot_error, label="actual")
plt.legend()
plt.show()
print(trot_error)

In [5]:
DATA_DIR = join(dirname(__file__), "..", "data")
TASK_DIR = "tfim"

CURR_DIR = DATA_DIR
check_and_make_dir(CURR_DIR)
CURR_DIR = join(CURR_DIR, TASK_DIR)
check_and_make_dir(CURR_DIR)

T = 2
J = 1
h = 1
gamma = 0.05
R = 128
n_p = 8
N_p = 2 ** n_p
print(f"N_p={N_p}")

error_tol = 5e-2
trotter_method = "second_order"

n_vals = np.arange(2, 10)

lchs_trotter_steps = np.zeros(len(n_vals), dtype=int)
lchs_two_qubit_gate_count_per_trotter_step = np.zeros(len(n_vals), dtype=int)
lchs_one_qubit_gate_count_per_trotter_step = np.zeros(len(n_vals), dtype=int)

schrodingerization_trotter_steps = np.zeros(len(n_vals), dtype=int)
schrodingerization_two_qubit_gate_count_per_trotter_step = np.zeros(len(n_vals), dtype=int)
schrodingerization_one_qubit_gate_count_per_trotter_step = np.zeros(len(n_vals), dtype=int)


'''Resource analysis for LCHS'''
print("Running resource analysis for LCHS")
for i, n in enumerate(n_vals):

    print(f"n={n}")
    k1 = R
    k2 = -R
    circuit = lchs_sim_circuit(n, k1, k2, T, J, h, gamma, r=1)

    np.savez(join(CURR_DIR, "lchs.npz"),
             lchs_trotter_steps=lchs_trotter_steps[:i+1],
             lchs_two_qubit_gate_count_per_trotter_step=lchs_two_qubit_gate_count_per_trotter_step[:i+1])

'''Resource analysis for Schrodingerization'''
print("\nRunning resource analysis for Schrodingerization")
for i, n in enumerate(n_vals):

    print(f"n={n}")

256
